In [23]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, rand, ntile, when, lit
from pyspark.sql.window import Window
from pyspark.sql.types import StructType, StructField, DoubleType
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.clustering import KMeans
import pandas as pd
from rdkit import Chem
from rdkit.Chem import Descriptors
from pyspark.sql.functions import pandas_udf
import pyspark.sql.functions as F
import findspark


import functions
import importlib
importlib.reload(functions)
from functions import *


In [9]:
spark.stop()

In [10]:
findspark.init()

In [11]:
spark = SparkSession.builder\
    .appName("Molecule Analysis")\
    .config("spark.executor.memory", "22g")\
    .config("spark.driver.memory", "16g")\
    .config("spark.memory.fraction", "0.7")\
    .config("spark.memory.storageFraction", "0.4")\
    .config("spark.network.timeout", "1200s")\
    .config("spark.executor.heartbeatInterval", "200s")\
    .config("spark.sql.broadcastTimeout", "1500s")\
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.sql.execution.arrow.maxRecordsPerBatch", "1000")\
    .getOrCreate()

In [12]:
filename = "leash-BELKA/train.parquet"
data = spark.read.parquet(filename)

data.printSchema()

data.show(5)

root
 |-- id: long (nullable = true)
 |-- buildingblock1_smiles: string (nullable = true)
 |-- buildingblock2_smiles: string (nullable = true)
 |-- buildingblock3_smiles: string (nullable = true)
 |-- molecule_smiles: string (nullable = true)
 |-- protein_name: string (nullable = true)
 |-- binds: long (nullable = true)

+---+---------------------+---------------------+---------------------+--------------------+------------+-----+
| id|buildingblock1_smiles|buildingblock2_smiles|buildingblock3_smiles|     molecule_smiles|protein_name|binds|
+---+---------------------+---------------------+---------------------+--------------------+------------+-----+
|  0| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|        BRD4|    0|
|  1| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|         HSA|    0|
|  2| C#CC[C@@H](CC(=O)...| C#CCOc1ccc(CN)cc1.Cl| Br.Br.NCC1CCCN1c1...|C#CCOc1ccc(CNc2nc...|         sEH|    0|
|  3|

In [13]:
binds_1_sampled = data.filter(col("binds") == 1).orderBy(rand()).limit(21000)
binds_0_sampled = data.filter(col("binds") == 0).orderBy(rand()).limit(2979000)

sampled_data = binds_0_sampled.union(binds_1_sampled)

sampled_data.groupBy("binds").count().show()

+-----+-------+
|binds|  count|
+-----+-------+
|    0|2979000|
|    1|  21000|
+-----+-------+



In [14]:
desc_schema = StructType([
    StructField("mol_wt", DoubleType(), True)
]
)

@pandas_udf(desc_schema)
def calculate_desc(smiles_series: pd.Series) -> pd.DataFrame:

    mols = smiles_series.apply(Chem.MolFromSmiles)
    
    mw = mols.apply(lambda mol: round(Descriptors.MolWt(mol), 3) if mol else None)


    return pd.DataFrame({'mol_wt': mw})
    
sampled_data_with_desc = sampled_data.withColumn("block1", calculate_desc(col("buildingblock1_smiles")))
sampled_data_with_desc = sampled_data_with_desc.withColumn("block3", calculate_desc(col("buildingblock3_smiles")))

In [16]:
selected_columns = [
    "id",
    col("block1.mol_wt").alias("block1_mol_wt"),
    col("block3.mol_wt").alias("block3_mol_wt"),
    "binds",
    ]

flattened_data = sampled_data_with_desc.select(*selected_columns)

desc_columns = [col_name for col_name in flattened_data.columns if col_name != "id" and col_name != "binds"]

assembler = VectorAssembler(inputCols=desc_columns, outputCol="features")
sampled_data_with_features = assembler.transform(flattened_data).select("id", "binds", "features")

In [22]:
num_bins = 30

# Create bins based on molecular weight
sampled_data_with_features = sampled_data_with_features.withColumn(
    "desc_bin", ntile(num_bins).over(Window.orderBy(col("features")))
)

In [26]:
bin_counts = sampled_data_with_features.groupBy("desc_bin", "binds").count()

# Pivot the table so each bin has counts for binds=0 and binds=1
bin_counts_pivot = bin_counts.groupBy("desc_bin").pivot("binds").sum("count").fillna(0)
bin_counts_pivot = bin_counts_pivot.withColumnRenamed("0", "count_binds_0").withColumnRenamed("1", "count_binds_1")

# Compute fraction to downsample binds=0 to match a 3:1 ratio
bin_counts_pivot = bin_counts_pivot.withColumn(
    "fraction_binds_0",
    (col("count_binds_1") * 3) / col("count_binds_0")  )


In [27]:
fractions = bin_counts_pivot.select("desc_bin", "fraction_binds_0").rdd.collectAsMap()

binds_0_sampled = sampled_data_with_features.filter(col("binds") == 0).sampleBy("desc_bin", fractions, seed=42)

# Keep 
# all binds=1 (since it’s the minority class)
binds_1 = sampled_data_with_features.filter(col("binds") == 1)

# Merge sampled binds=0 with all binds=1
balanced_data = binds_1.unionByName(binds_0_sampled.select(binds_1.columns))

25/02/12 13:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 13:51:46 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 13:53:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 13:53:52 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 13:53:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 13:53:53 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 1

In [28]:
balanced_data.printSchema()

root
 |-- id: long (nullable = true)
 |-- binds: long (nullable = true)
 |-- features: vector (nullable = true)
 |-- desc_bin: integer (nullable = false)



In [29]:
balanced_data.write.save("./intermediates/balanced_data_3_mln", format="parquet", mode='overwrite')

25/02/12 22:04:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 22:04:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 22:04:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 22:04:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 22:04:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 22:06:26 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/12 2

In [ ]:
data_subset = data.select("id", "molecule_smiles")

final_data = balanced_data.drop("features").join(data_subset, on="id", how="left")
final_data.write.save("./intermediates/final_data_3_mln", format="parquet", mode='overwrite')

25/02/13 05:57:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/13 05:57:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/13 05:57:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/13 05:57:24 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/13 05:59:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/13 05:59:43 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/02/13 0

In [ ]:
from lightgbm import LGBMClassifier

final_data = final_data.toPandas()

fpg = rdFingerprintGenerator.GetMorganGenerator(radius=3, fpSize=700, useBondTypes=True, includeChirality=True, includeRingMembership=True)

def compute_fn(self, smiles):
        
        mol = Chem.MolFromSmiles(smiles)
        if mol is None:
            return np.zeros(700, dtype=np.int8)
        
        ao = rdFingerprintGenerator.AdditionalOutput()
        ao.AllocateBitInfoMap()
        
        
        fp = self.fpg.GetCountFingerprint(mol, additionalOutput=ao)

        bit_info = ao.GetBitInfoMap()
        arr = np.zeros(700, dtype=np.int8)
        DataStructs.ConvertToNumpyArray(fp, arr)
    
        return arr, bit_info



results = final_data['molecule_smiles'].apply(compute_fn)
final_data['fp'] = results.apply(lambda x: x[0])
bit_infos = results.apply(lambda x: x[1]).tolist()
protein_dummies = pd.get_dummies(final_data['protein_name'], prefix='protein')
final_data.drop(['molecule_smiles', 'buildingblock1_smiles', 'buildingblock2_smiles', 'buildingblock3_smiles'], axis=1, inplace=True)
fingerprint_df = pd.DataFrame(final_data["fp"].to_list(), index=final_data.index)
final_data = pd.concat([final_data.drop(columns=["fp", "protein_name"]), fingerprint_df, protein_dummies], axis=1)

feature_cols = [col for col in final_data.columns if col not in ['id', 'binds']]

X = final_data[feature_cols]
y = final_data['binds']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

lgb_cls = LGBMClassifier()
lgb_cls.fit(X_train, y_train)

y_pred = lgb_cls.predict(X_test)
accuracy = accuracy_score(y_test, y_pred)
f1 = f1_score(y_test, y_pred)
print(f'Accuracy: {accuracy:.4f}, F1: {f1:.4f}')
